# Import the necessary library

In [1]:
import pandas as pd
import os
from datetime import datetime, timedelta
import os
import pyarrow as pa
import pyarrow.parquet as pq

# THE FUNCTION BELOW READS THROUGH ALL THE CSV FILES IN OUR LOCAL ENVIRONMENT

In [ ]:
def load_and_combine_csv(start_date, end_date, file_directory, output_file):
    date_range = pd.date_range(start=start_date, end=end_date)

    # Pass 1: collect union of columns across all files
    print("Pass 1: scanning column headers...")
    all_columns = []
    seen = set()
    valid_files = []

    for single_date in date_range:
        file_name = single_date.strftime("%Y-%m-%d") + '.csv'
        file_path = os.path.join(file_directory, file_name)

        if not os.path.exists(file_path):
            continue

        # Read only the header — fast and memory-light
        cols = pd.read_csv(file_path, nrows=0).columns.tolist()
        for c in cols:
            if c not in seen:
                seen.add(c)
                all_columns.append(c)
        valid_files.append((single_date, file_path, file_name))

    print(f"Found {len(all_columns)} unique columns across {len(valid_files)} files")

    # Pass 2: stream each file, align to union schema, write to parquet
    print("Pass 2: writing parquet...")
    loaded_files, empty_files = [], []
    writer = None

    try:
        for single_date, file_path, file_name in valid_files:
            df = pd.read_csv(file_path)

            if df.empty:
                empty_files.append(file_name)
                continue

            # Add missing columns as NaN, reorder to match union
            df = df.reindex(columns=all_columns)

            table = pa.Table.from_pandas(df, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_file, table.schema, compression='snappy')

            writer.write_table(table)
            loaded_files.append(file_name)
            del df, table
    finally:
        if writer is not None:
            writer.close()

    missing_count = len(date_range) - len(valid_files)
    print(f"Combined file created: {output_file}")
    print(f"Loaded files: {len(loaded_files)}")
    print(f"Missing files: {missing_count}")
    print(f"Empty files: {len(empty_files)}")

start_date = '2025-07-01'
end_date = '2025-09-30'
file_directory = '/Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q3_2025'
output_file = '/Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q3_2025/Q3_2025_data.parquet'

load_and_combine_csv(start_date, end_date, file_directory, output_file)

Pass 1: scanning column headers...
Found 197 unique columns across 92 files
Pass 2: writing parquet...
Combined file created: /Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q3_2025/Q3_2025_data.parquet
Loaded files: 92
Missing files: 0
Empty files: 0


# Checking if the data contains records for all the months

In [ ]:

def check_for_q1_records(file_path):
    # Read the parquet schema without loading data
    parquet_file = pq.ParquetFile(file_path)
    columns = parquet_file.schema.names

    # Find the date column
    date_column = next((c for c in columns if 'date' in c.lower()), None)
    if date_column is None:
        print("No date column found in the file.")
        return

    # Load only the date column for the entire file
    df_date = pd.read_parquet(file_path, columns=[date_column])

    # Ensure datetime dtype
    if not pd.api.types.is_datetime64_any_dtype(df_date[date_column]):
        df_date[date_column] = pd.to_datetime(df_date[date_column], errors='coerce')

    df_date.dropna(subset=[date_column], inplace=True)

    date_min = df_date[date_column].min()
    date_max = df_date[date_column].max()
    print(f"The date range in the file is from {date_min.date()} to {date_max.date()}")

    # Check Q1 bounds
    q1_start = pd.Timestamp('2025-07-01')
    q1_end = pd.Timestamp('2025-09-30')
    if date_min >= q1_start and date_max <= q1_end:
        print("The file contains records for Q3.")
    else:
        print("The file does not contain records exclusively for Q3.")

    # Extra: show what days are actually present (useful for spotting gaps)
    unique_days = df_date[date_column].dt.normalize().nunique()
    print(f"Unique days in file: {unique_days}")

file_path = '/Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q3_2025/Q3_2025_data.parquet'
check_for_q1_records(file_path)

The date range in the file is from 2025-07-01 to 2025-09-30
The file contains records for Q3.
Unique days in file: 92


In [17]:

file_path = '/Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q3_2025/Q3_2025_data.parquet'

# Option 1: Just the metadata — no data loaded, instant
pf = pq.ParquetFile(file_path)
print(f"Rows: {pf.metadata.num_rows:,}")
print(f"Columns: {pf.metadata.num_columns}")
print(f"Row groups: {pf.metadata.num_row_groups}")
print(f"File size on disk: ~{pf.metadata.serialized_size / 1e6:.1f} MB metadata")
print(f"\nColumn names:\n{pf.schema.names}")

Rows: 29,844,451
Columns: 197
Row groups: 92
File size on disk: ~2.4 MB metadata

Column names:
['date', 'serial_number', 'model', 'capacity_bytes', 'failure', 'datacenter', 'cluster_id', 'vault_id', 'pod_id', 'pod_slot_num', 'is_legacy_format', 'smart_1_normalized', 'smart_1_raw', 'smart_2_normalized', 'smart_2_raw', 'smart_3_normalized', 'smart_3_raw', 'smart_4_normalized', 'smart_4_raw', 'smart_5_normalized', 'smart_5_raw', 'smart_7_normalized', 'smart_7_raw', 'smart_8_normalized', 'smart_8_raw', 'smart_9_normalized', 'smart_9_raw', 'smart_10_normalized', 'smart_10_raw', 'smart_11_normalized', 'smart_11_raw', 'smart_12_normalized', 'smart_12_raw', 'smart_13_normalized', 'smart_13_raw', 'smart_15_normalized', 'smart_15_raw', 'smart_16_normalized', 'smart_16_raw', 'smart_17_normalized', 'smart_17_raw', 'smart_18_normalized', 'smart_18_raw', 'smart_22_normalized', 'smart_22_raw', 'smart_23_normalized', 'smart_23_raw', 'smart_24_normalized', 'smart_24_raw', 'smart_27_normalized', 'smart

In [18]:
# Option 2: First N rows only — small memory footprint
# iter_batches yields chunks; grab the first one and stop
batch = next(pf.iter_batches(batch_size=1000))
df_head = batch.to_pandas()
print(df_head.head())
print(df_head.dtypes)

         date serial_number                 model  capacity_bytes  failure  \
0  2025-07-01  2207E60CC65A        CT250MX500SSD1    250059350016        0   
1  2025-07-01  2340E87B92B5        CT250MX500SSD1    250059350016        0   
2  2025-07-01  2340E87B97E8        CT250MX500SSD1    250059350016        0   
3  2025-07-01      2EGK64VX  HGST HUH728080ALE604   8001563222016        0   
4  2025-07-01      2EHZAKAX  HGST HUH728080ALE604   8001563222016        0   

  datacenter  cluster_id  vault_id  pod_id  pod_slot_num  ...  \
0       sac0           0      1028      13           NaN  ...   
1       sac0           0      1028      14           NaN  ...   
2       sac0           0      1028       2           NaN  ...   
3       sac0           0      1028       4          12.0  ...   
4       sac0           0      1028      12          30.0  ...   

   smart_250_normalized  smart_250_raw  smart_251_normalized  smart_251_raw  \
0                   NaN            NaN                   NaN 

In [25]:
df_head['failure'].unique()

array([0])

1254 failures in 29,844,451 rows (0.0042%)


In [26]:
def combine_parquets(input_paths, output_file):
    # Pass 1: union of columns across all input files
    all_columns = []
    seen = set()
    for path in input_paths:
        cols = pq.ParquetFile(path).schema.names
        for c in cols:
            if c not in seen:
                seen.add(c)
                all_columns.append(c)
    print(f"Union schema: {len(all_columns)} columns")

    # Pass 2: stream batches from each file, align to union, write
    writer = None
    total_rows = 0
    try:
        for path in input_paths:
            pf = pq.ParquetFile(path)
            print(f"Processing {path} ({pf.metadata.num_rows:,} rows)")
            for batch in pf.iter_batches(batch_size=100_000):
                df = batch.to_pandas()
                df = df.reindex(columns=all_columns)  # fill missing cols with NaN
                table = pa.Table.from_pandas(df, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(output_file, table.schema, compression='snappy')
                writer.write_table(table)
                total_rows += len(df)
                del df, table
    finally:
        if writer is not None:
            writer.close()

    print(f"Wrote {total_rows:,} rows to {output_file}")

base = '/Users/danielabutu/Desktop/AI and ML file/HDD journals'
combine_parquets(
    input_paths=[
        f'{base}/data_Q1_2025/Q1_2025_data.parquet',
        f'{base}/data_Q2_2025/Q2_2025_data.parquet',
        f'{base}/data_Q3_2025/Q3_2025_data.parquet',
    ],
    output_file=f'{base}/train_Q1_Q3_2025.parquet',
)

Union schema: 197 columns
Processing /Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q1_2025/Q1_2025_data.parquet (27,799,986 rows)
Processing /Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q2_2025/Q2_2025_data.parquet (28,808,042 rows)
Processing /Users/danielabutu/Desktop/AI and ML file/HDD journals/data_Q3_2025/Q3_2025_data.parquet (29,844,451 rows)
Wrote 86,452,479 rows to /Users/danielabutu/Desktop/AI and ML file/HDD journals/train_Q1_Q3_2025.parquet


In [27]:
pf = pq.ParquetFile('/Users/danielabutu/Desktop/AI and ML file/HDD journals/train_Q1_Q3_2025.parquet')
total, failures = 0, 0
for batch in pf.iter_batches(batch_size=200_000, columns=['failure']):
    df = batch.to_pandas()
    total += len(df)
    failures += int(df['failure'].sum())
print(f"{failures} failures in {total:,} rows ({100*failures/total:.4f}%)")

3382 failures in 86,452,479 rows (0.0039%)


In [28]:
failures

3382